# Submission 2 - 234-class multi-label inference

Forked from `birdclef-2026-submission-1.ipynb`. Uses the multi-label model trained in `birdclef_plus_2026_multilabel_attempt.ipynb`, so all 234 columns (including the 28 species missing from `train_audio`) get real probabilities instead of being zero-filled.

In [ ]:
import json
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from fastai.vision.all import load_learner


In [2]:
import kagglehub
path = kagglehub.competition_download('birdclef-2026')
print('Path to competition files:', path)

Path to competition files: /kaggle/input/competitions/birdclef-2026


In [3]:
MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/1/model_multilabel_234.pkl'
VOCAB_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/1/vocab.json'
TEST_DIR = Path(path) / 'test_soundscapes'
TARGET_SIZE = (224, 224)
CLIP_DURATION = 5
SAMPLE_RATE = 32000
BATCH_SIZE = 64

In [ ]:
sample_sub = pd.read_csv(Path(path) / 'sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']
assert len(all_species) == 234, f'expected 234, got {len(all_species)}'

learn = load_learner(MODEL_PATH, cpu=False)
model_vocab = list(learn.dls.vocab)

if Path(VOCAB_PATH).exists():
    saved_vocab = json.load(open(VOCAB_PATH))
    assert saved_vocab == model_vocab, 'vocab.json disagrees with learner.dls.vocab'

assert model_vocab == all_species, (
    'Model vocab does not match sample_submission column order. '
    f'len(model_vocab)={len(model_vocab)}, len(all_species)={len(all_species)}; '
    f'first mismatch at {next((i for i, (a, b) in enumerate(zip(model_vocab, all_species)) if a != b), None)}'
)
print('Vocab order matches sample_submission for all 234 classes.')

In [ ]:
from torchvision import transforms as T

def audio_to_pil(samples: np.ndarray) -> Image.Image:
    s = librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE)
    s_db = librosa.power_to_db(s, ref=np.max)
    s_min, s_max = float(s_db.min()), float(s_db.max())
    if s_max == s_min:
        s_norm = np.zeros_like(s_db, dtype=np.uint8)
    else:
        s_norm = ((s_db - s_min) / (s_max - s_min) * 255).astype(np.uint8)
    return Image.fromarray(s_norm).resize(TARGET_SIZE).convert('RGB')

tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

learn.model.eval()
device = next(learn.model.parameters()).device

test_files = sorted(TEST_DIR.glob('*.ogg'))
if not test_files:
    fallback_dir = Path(path) / 'train_soundscapes'
    test_files = sorted(fallback_dir.glob('*.ogg'))[:3]
    print(f'[dry-run] test_soundscapes empty, using {len(test_files)} train_soundscapes files')
else:
    print(f'Found {len(test_files)} test soundscape files')

clip_length = CLIP_DURATION * SAMPLE_RATE
row_ids = []
all_preds = []

for soundscape in test_files:
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    file_imgs = []
    file_row_ids = []
    start = 0
    end_time = CLIP_DURATION
    while start + clip_length <= len(samples):
        chunk = samples[start:start + clip_length]
        file_imgs.append(audio_to_pil(chunk))
        file_row_ids.append(f'{soundscape.stem}_{end_time}')
        start += clip_length
        end_time += CLIP_DURATION
    for i in range(0, len(file_imgs), BATCH_SIZE):
        batch = torch.stack([tfm(img) for img in file_imgs[i:i + BATCH_SIZE]]).to(device)
        with torch.no_grad():
            logits = learn.model(batch)
        all_preds.append(torch.sigmoid(logits).cpu())
    row_ids.extend(file_row_ids)

preds_np = torch.cat(all_preds).numpy()
assert preds_np.shape == (len(row_ids), 234), preds_np.shape
print(f'Inference complete: {preds_np.shape[0]} clips across {len(test_files)} files')


In [7]:
submission = pd.DataFrame(preds_np, columns=all_species)
submission.insert(0, 'row_id', row_ids)
assert list(submission.columns) == ['row_id'] + all_species
submission.to_csv('submission.csv', index=False)
print(f'Done. {len(submission)} rows written.')
submission.head()

Done. 36 rows written.


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.005991,0.953403,0.004476,0.033234,0.125475,0.002555,0.000216,0.003880,0.006975,...,0.014109,0.003302,0.347309,0.003286,0.001227,0.008704,0.000706,0.103584,0.001823,0.197344
1,BC2026_Train_0001_S08_20250606_030007_10,0.004931,0.956036,0.003438,0.038782,0.178847,0.002312,0.000156,0.003527,0.005309,...,0.010556,0.002103,0.435344,0.003268,0.001185,0.009665,0.001066,0.126758,0.002450,0.193407
2,BC2026_Train_0001_S08_20250606_030007_15,0.005059,0.929776,0.003544,0.027210,0.176613,0.001935,0.000250,0.003104,0.004493,...,0.018875,0.003106,0.529575,0.002938,0.000814,0.012141,0.000845,0.112905,0.003352,0.266118
3,BC2026_Train_0001_S08_20250606_030007_20,0.005615,0.946670,0.002969,0.024503,0.085494,0.002034,0.000215,0.003444,0.005527,...,0.020779,0.002689,0.351254,0.003142,0.001002,0.009773,0.000755,0.086178,0.001185,0.198932
4,BC2026_Train_0001_S08_20250606_030007_25,0.002892,0.928922,0.003080,0.032848,0.078452,0.001440,0.000117,0.002075,0.004807,...,0.008609,0.003003,0.403895,0.001849,0.000732,0.009179,0.000683,0.076557,0.000795,0.209462
